In [1]:
import pandas as pd
import re
from modules.Corpus import Corpus
from modules.Document import Document

# Chargement des données
df = pd.read_csv('data/discours_US.csv', sep='\t')

# Création de l'objet Corpus
mon_corpus = Corpus("Discours_US")

# Ajout des documents au corpus en découpant les discours en phrases
from tqdm import tqdm
for index, row in tqdm(df.iterrows(), total=len(df), desc="Traitement des discours"):
    # Découpage du texte en phrases
    # On utilise une expression régulière pour découper selon la ponctuation terminale
    phrases = re.split(r'(?<=[.!?]) +', str(row['text']))
    
    for i, phrase in enumerate(phrases):
        phrase = phrase.strip()
        if phrase:  # On n'ajoute que les phrases non vides
            # Création d'un titre pour chaque phrase (Titre original + index de la phrase)
            titre = f"{row['descr']} - Phrase {i+1}"
            
            # Création de l'objet Document
            doc = Document(
                titre=titre,
                auteur=row['speaker'],
                date=row['date'],
                url=row['link'],
                texte=phrase
            )
            
            # Ajout au corpus
            mon_corpus.add_document(doc)

# Affichage des statistiques du corpus pour vérification
print(mon_corpus)
mon_corpus.stats(n=15)


Traitement des discours: 100%|██████████| 164/164 [00:00<00:00, 970.02it/s]


Corpus: Discours_US
  - Nombre de documents: 32925
  - Nombre d'auteurs: 2
  - ID du prochain document: 32925

STATISTIQUES DU CORPUS 'Discours_US'

📊 Nombre de documents: 32925
📚 Nombre de mots différents dans le corpus: 12203
📝 Nombre total de mots: 514320

🏆 TOP 15 MOTS LES PLUS FRÉQUENTS

    nombre_document   mot  occurrences  document_frequency
0             32925   the        20250               12960
1             32925    to        18790               12746
2             32925   and        18490               12943
3             32925     i        12339                9123
4             32925    of        10569                8240
5             32925    we         9713                7700
6             32925  that         9151                7484
7             32925     a         9137                7393
8             32925    in         8045                6629
9             32925   you         7269                5507
10            32925    it         6476                557

,nombre_document,mot,occurrences,document_frequency
0,32925,the,20250,12960
1,32925,to,18790,12746
2,32925,and,18490,12943
3,32925,i,12339,9123
4,32925,of,10569,8240
...,...,...,...,...
12198,32925,richest,1,1
12199,32925,contracted,1,1
12200,32925,overwhelm,1,1
12201,32925,considerate,1,1


In [2]:
# Tests des fonctions de recherche du Corpus
print("\n--- TEST RECHERCHE (search) ---")
keyword = "America"
matches = list(mon_corpus.search(keyword))
print(f"Nombre d'occurrences de '{keyword}': {len(matches)}")



--- TEST RECHERCHE (search) ---
Nombre d'occurrences de 'America': 3220


In [3]:
# Tests de la fonction concorde
print("\n--- TEST CONCORDANCIER (concorde) ---")
expression = "democracy"
df_concorde = mon_corpus.concorde(expression, context_size=30)
print(f"Concordancier pour '{expression}' (5 premiers résultats) :")
print(df_concorde.head())



--- TEST CONCORDANCIER (concorde) ---
Concordancier pour 'democracy' (5 premiers résultats) :
                  contexte gauche motif trouvé                  contexte droit
0  th the kind of assault on our     democracy  , on voting rights, and on the
1  be they can understand in our     democracy  , you know, we do try to close
2  donesia is a relatively young     democracy  . So I said, \"You're right, w
3   young people more engaged in     democracy  , not less. In fact I would sa
4  the greatest, longest-lasting     democracy   in the history of the world. 


In [4]:
# Utilisation du moteur de recherche avancé
from modules.SearchEngine import SearchEngine

print("\n--- TEST MOTEUR DE RECHERCHE (TF-IDF) ---")
moteur = SearchEngine(mon_corpus)
query = "economic growth and jobs"
resultats = moteur.search(query, n_results=5)
print(f"Résultats pour la requête : '{query}'")
print(resultats[['Rank', 'Score', 'Title', 'Text_Preview']])




--- TEST MOTEUR DE RECHERCHE (TF-IDF) ---
      → 12203 mots uniques indexés.
      → Construction de la matrice TF...


Indexation: 100%|██████████| 32925/32925 [00:00<00:00, 92504.68it/s]


      → Matrice TF 32925 × 12203 (creuse) construite.
      → Matrice TF-IDF 32925 × 12203 (creuse).
      → IDF: min=0.9324, max=10.4020
      → Requête vectorisée en 12203 dimensions (TF-IDF).


Récupération des résultats: 100%|██████████| 5/5 [00:00<00:00, 3484.22it/s]

Résultats pour la requête : 'economic growth and jobs'
   Rank     Score                                              Title  \
0     1  0.585738           Interview with Charlie Rose - Phrase 368   
1     2  0.527450      Debate between Trump and Clinton - Phrase 119   
2     3  0.509425      Debate between Trump and Clinton - Phrase 354   
3     4  0.500524      Debate between Trump and Clinton - Phrase 357   
4     5  0.486463  Remarks at Toyota of Portsmouth in Portsmouth,...   

                                        Text_Preview  
0  That's why when I rolled out my economic plans...  
1  But it's because I see this—we need to have st...  
2                                 There's no growth.  
3                       And that's, like, no growth.  
4  It's the most pro-growth economic plan in Amer...  
